# Acknowledgements benchmark

Create benchmark dataset with new answer format:
- start from eval dataset dataesr/acknowledgments_json
- make prediction with mistral for new format
- manually validate entries



## Get eval dataset

In [1]:
from datasets import load_dataset

dataset = load_dataset("dataesr/acknowledgments_json", split="eval")
df = dataset.to_pandas()

In [2]:
df

,id,input,completion
0,doi10.1002/cbic.202300093,Acknowledgements We thank The Company of Biol...,"{""entities"": [{""entity"": ""Science"", ""entity_ty..."
1,doi10.1002/jsfa.12651,ACKNOWLEDGEMENTS The authors are grateful to t...,"{""entities"": [{""entity"": ""BMGF (Bill & Melinda..."
2,doi10.1007/jhep06(2023)018,Acknowledgments We thank S. Sethi for discus...,"{""entities"": [{""entity"": ""ISF"", ""entity_type"":..."
3,doi10.1007/jhep06(2023)188,Acknowledgements We thank CERN for the very ...,"{""entities"": [{""entity"": ""Wallenberg Foundatio..."
4,doi10.1007/jhep10(2023)001,Acknowledgements We thank CERN for the very ...,"{""entities"": [{""entity"": ""BSF-NSF"", ""entity_ty..."
...,...,...,...
95,doi10.3847/2041-8213/acc9c8,Acknowledgments J.Y. and X.F. acknowledge supp...,"{""entities"": [{""entity"": ""Independent Research..."
96,doi10.5194/amt-16-4899-2023,Acknowledgements.We gratefully thank the Strat...,"{""entities"": [{""entity"": ""ANR"", ""entity_type"":..."
97,doi10.5194/egusphere-2023-103,"Acknowledgment JZ, ZZ and AG are supported by ...","{""entities"": [{""entity"": ""NSF"", ""entity_type"":..."
98,doi10.5194/ejm-35-219-2023,Acknowledgements.Sylvie Demouchy thanks Chris...,"{""entities"": [{""entity"": ""ERC"", ""entity_type"":..."


## Get completion

In [29]:
# system prompt
SYSTEM_PROMPT = """You are an expert annotator for scientific acknowledgement sections. Your task is to produce a structured extraction from acknowledgement text.

You will receive the raw acknowledgement text. Analyze it carefully, then produce a JSON annotation.

## Entity types to extract

### Funders
Organizations that provide financial support. Extract:
- `mention`: the exact surface form as it appears in text
- `canonical_name`: the standard/official name of the funder (e.g. "Agence Nationale de la Recherche" not "French ANR")
- `funder_short`: common abbreviation (e.g. "ANR", "NSF", "ERC", "DFG")
- `country`: ISO 2-letter code (FR, US, EU, DE, UK, CN, JP, etc.)
- `grant_ids`: list of grant/award identifiers associated with THIS specific funder. Only attribute a grant ID to the funder it belongs to — do not duplicate the same ID across multiple funders unless the text explicitly associates it with each.
- `programs`: list of program/scheme names through which this funder provided support (e.g. "Horizon 2020", "ERC Starting Grant", "KAKENHI", "CPER", "FEDER")

Important distinctions:
- The **funder** is the organization that controls the money. A **program** is the mechanism through which they fund. Always identify both separately.
- **European Commission (EC)** is the funder for Horizon 2020, FP7, FP6, Horizon Europe, MSCA, FEDER, etc. These are programs, not funders.
- **ANR** is the funder for Labex, Equipex, Idex (Investissements d'Avenir programs). List them as programs under ANR.
- **CPER** (Contrat de Plan État-Région) and **FEDER** (Fonds Européen de Développement Régional) are specific funding instruments — list them as programs under the appropriate funder (French state / regional council for CPER, EC for FEDER).
- A **university or lab** giving internal funding IS a funder in that context.
- **CNRS, INSERM, INRAE** can be funders (when they fund) or affiliations (when they host). Classify based on context: "funded by CNRS" = funder; "CNRS laboratory" = institution.
- When a person from a company supervises analyses or provides technical services, that person goes in `persons_thanked` with role `technical_assistance`, not `supervision`.

### Projects
Named research projects mentioned in the acknowledgement. These are NOT funders — they are funded activities. Extract:
- `mention`: exact surface form (e.g. "City4Age", "STEALTH", "BENEFIC", "amePLM")
- `name`: standardized project name
- `funder`: which funder funds this project, if identifiable from context
- `grant_id`: the grant ID associated with this project, if mentioned

This is important: many acknowledgements name projects that are the vehicle for funding. Capture them here rather than losing them or misclassifying them as funders.

Note: a project name can also reveal a funder not mentioned explicitly. For example, "RAPID FLEXyRADIO" implies DGA (Direction Générale de l'Armement) funding because RAPID is a DGA program. When you can identify the funder behind a program/project, add that funder to the `funders` list AND capture the project here. Do not drop funders just because they are only implied through a project/program name.

### Infrastructure
Computing resources, experimental facilities, instruments, observatories. Extract:
- `mention`: exact surface form
- `name`: standardized name
- `type`: one of `computing` | `experimental_facility` | `instrument` | `observatory` | `biobank` | `other`
- `resource_id`: any allocation/project ID (e.g. GENCI allocation numbers)

### Persons thanked
People acknowledged by name (not as authors). Extract:
- `name`: as written in text
- `role`: what they're thanked for. One of: `technical_assistance` | `discussions` | `review` | `language_editing` | `data_resources` | `supervision` | `administrative` | `other`
- `affiliation`: if mentioned

Role guidelines:
- `technical_assistance`: lab work, sample preparation, data acquisition, instrument operation, running analyses, field work, geomatics, IT support
- `discussions`: scientific discussions, comments, remarks, insights
- `review`: reviewing the manuscript, referee comments
- `language_editing`: proofreading, English editing, language revision
- `data_resources`: providing data, samples, reagents, strains, specimens
- `supervision`: PhD/postdoc supervision, mentoring (academic context only — a company person overseeing analyses is `technical_assistance`)
- `administrative`: administrative support, project management

### Private companies
Corporate entities mentioned. Extract:
- `mention`: exact surface form
- `name`: standardized name
- `role`: `funding` | `data_provider` | `partnership` | `employment` | `other`

## Output format

Return ONLY a JSON object with this structure:
```json
{
  "funders": [...],
  "projects": [...],
  "infrastructure": [...],
  "persons_thanked": [...],
  "private_companies": [...],
}
```

Empty lists are fine when no entities of that type are present."""

In [4]:
def build_user_prompt(text: str) -> str:
    return f"""Annotate this acknowledgement text:

<text>
{text}
</text>"""

In [14]:
from mistralai.client import Mistral
from dotenv import load_dotenv
import os
import json

load_dotenv()

mistral = Mistral(api_key=os.getenv("MISTRAL_API_KEY"))


def annotate_one(text: str) -> dict:
    try:
        chat_response = mistral.chat.complete(
            model="mistral-large-latest",
            messages=[{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": build_user_prompt(text)}],
            temperature=0.2,
            response_format={
                "type": "json_object",
            },
        )
        text_response = chat_response.choices[0].message.content
        json_response = json.loads(text_response)
        return json_response
    except Exception as error:
        return {"error": str(error)}

Annotate and verify one entry

In [24]:
sample = df.sample(1, random_state=0)["input"].iloc[0]
print(sample)

Declaration of competing interest The authors declare the following financial interests/personal relationships which may be considered as potential competing interests: Frank Rasche  reports financial support was provided by  University of Hohenheim Institute of Agricultural Sciences in the Tropics . 
 Acknowledgements This field study has been made possible by the  Swedish Infrastructure for Ecosystem Science (SITES) , in this case,  SITES Lönnstorp Research Station  at SLU. SITES receives funding through the  Swedish Research Council  under the grant no  2017-00635 ). We thank the Perennial grain group from  AGROPOLE-ISARA LYON (Institut supérieur d'agriculture Rhône-Alpes)  and  Patrice Barrey  for the opportunity to use their fields. Furthermore, this work was partly supported by the  Fonds de la Recherche Scientifique -FNRS - under grant n •  R.8003.20 . We would like to thank  Pierre Aubry ,  Camille Bathellier  and  Thorsten Ruf  for their help on the field. Special thanks to  M

In [30]:
annotation = annotate_one(sample)
annotation

{'funders': [{'mention': 'University of Hohenheim Institute of Agricultural Sciences in the Tropics',
   'canonical_name': 'University of Hohenheim Institute of Agricultural Sciences in the Tropics',
   'funder_short': None,
   'country': 'DE',
   'grant_ids': [],
   'programs': []},
  {'mention': 'Swedish Research Council',
   'canonical_name': 'Swedish Research Council',
   'funder_short': 'VR',
   'country': 'SE',
   'grant_ids': ['2017-00635'],
   'programs': []},
  {'mention': 'Fonds de la Recherche Scientifique -FNRS',
   'canonical_name': 'Fonds de la Recherche Scientifique - FNRS',
   'funder_short': 'FNRS',
   'country': 'BE',
   'grant_ids': ['R.8003.20'],
   'programs': []},
  {'mention': 'German Research Foundation',
   'canonical_name': 'German Research Foundation',
   'funder_short': 'DFG',
   'country': 'DE',
   'grant_ids': ['RA 1717/8-1'],
   'programs': []},
  {'mention': 'Bio-divERsA joint call for research proposals',
   'canonical_name': 'European Commission',
   '

In [ ]:
corrected = {'funders': [{'mention': 'University of Hohenheim Institute of Agricultural Sciences in the Tropics',
   'canonical_name': 'University of Hohenheim Institute of Agricultural Sciences in the Tropics',
   'funder_short': None,
   'country': 'DE',
   'grant_ids': [],
   'programs': []},
  {'mention': 'Swedish Research Council',
   'canonical_name': 'Swedish Research Council',
   'funder_short': 'VR',
   'country': 'SE',
   'grant_ids': ['2017-00635'],
   'programs': []},
  {'mention': 'Fonds de la Recherche Scientifique -FNRS',
   'canonical_name': 'Fonds de la Recherche Scientifique',
   'funder_short': 'FNRS',
   'country': 'BE',
   'grant_ids': ['R.8003.20'],
   'programs': []},
  {'mention': 'German Research Foundation',
   'canonical_name': 'German Research Foundation',
   'funder_short': 'DFG',
   'country': 'DE',
   'grant_ids': ['RA 1717/8-1'],
   'programs': []},
  {'mention': 'Bio-divERsA joint call for research proposals',
   'canonical_name': 'European Commission',
   'funder_short': 'EC',
   'country': 'EU',
   'grant_ids': [],
   'programs': ['BiodivClim ERA-Net COFUND programme']}],
 'projects': [{'mention': 'NAPERDIV',
   'name': 'NAPERDIV',
   'funder': 'European Commission',
   'grant_id': None}],
 'infrastructure': [{'mention': 'Swedish Infrastructure for Ecosystem Science (SITES)',
   'name': 'Swedish Infrastructure for Ecosystem Science',
   'type': 'experimental_facility',
   'resource_id': None},
  {'mention': 'SITES Lönnstorp Research Station',
   'name': 'SITES Lönnstorp Research Station',
   'type': 'experimental_facility',
   'resource_id': None}],
 'persons_thanked': [{'name': 'Patrice Barrey',
   'role': 'technical_assistance',
   'affiliation': "AGROPOLE-ISARA LYON (Institut supérieur d'agriculture Rhône-Alpes)"},
  {'name': 'Pierre Aubry',
   'role': 'technical_assistance',
   'affiliation': None},
  {'name': 'Camille Bathellier',
   'role': 'technical_assistance',
   'affiliation': None},
  {'name': 'Thorsten Ruf',
   'role': 'technical_assistance',
   'affiliation': None},
  {'name': 'Mickaël Hedde', 'role': 'data_resources', 'affiliation': None},
  {'name': 'Elvira Sieberger',
   'role': 'technical_assistance',
   'affiliation': None},
  {'name': 'Petra Ziegler',
   'role': 'technical_assistance',
   'affiliation': None}],
 'private_companies': [{'mention': "AGROPOLE-ISARA LYON (Institut supérieur d'agriculture Rhône-Alpes)",
   'name': 'AGROPOLE-ISARA LYON',
   'role': 'partnership'}]}